In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/Readme.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/Supplementary.pdf
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/D09_SA01_R02.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/D13_SA01_R03.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/D11_SA01_R04.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/D12_SA01_R04.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/D13_SA01_R01.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/F09_SA01_R03.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/F09_SA01_R02.txt
/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/SA01/F14_SA01_R02.txt
/kaggle/input/d

In [3]:
import os

path = '/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset'
for item in os.listdir(path):
    print(item)

SisFall_dataset


In [ ]:
import numpy as np
import pandas as pd
import os, glob, pickle, json, shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization,
                                     MaxPooling1D, Dropout, LSTM, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow.keras.backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, recall_score,
                             precision_score, accuracy_score, roc_curve)

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# ── CONFIG ───────────────────────────────────────────────────
SISFALL_DIR = '/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset'
OUTPUT_DIR  = '/kaggle/working/'
ORIG_HZ     = 200
TARGET_HZ   = 50
WINDOW_SIZE = 250
STEP_SIZE   = 50
N_FEATURES  = 6
BATCH_SIZE  = 64
MAX_EPOCHS  = 50
PATIENCE    = 10
VAL_SPLIT   = 0.10

print(f"{'='*55}")
print(f"  DATASET  : SisFall")
print(f"  FEATURES : Acc1(xyz) + Gyr(xyz) = {N_FEATURES}")
print(f"  SAMPLING : {ORIG_HZ}Hz → {TARGET_HZ}Hz")
print(f"  WINDOW   : {WINDOW_SIZE} samples ({WINDOW_SIZE/TARGET_HZ:.1f}s)")
print(f"  OVERLAP  : 80%")
print(f"{'='*55}")

# ── HELPERS ──────────────────────────────────────────────────
def resample_signal(signal, orig_hz, target_hz):
    orig_len   = len(signal)
    target_len = int(orig_len * target_hz / orig_hz)
    if target_len < 2:
        return signal
    orig_t   = np.linspace(0, 1, orig_len)
    target_t = np.linspace(0, 1, target_len)
    if signal.ndim == 1:
        return np.interp(target_t, orig_t, signal)
    return np.column_stack([
        np.interp(target_t, orig_t, signal[:, i])
        for i in range(signal.shape[1])
    ])

def sliding_window(signal, window_size, step_size):
    windows, start = [], 0
    while start + window_size <= signal.shape[0]:
        windows.append(signal[start:start + window_size])
        start += step_size
    return windows

def load_sisfall_file(filepath):
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
        rows = []
        for line in lines:
            line = line.strip().rstrip(';').strip()
            if not line:
                continue
            vals = [float(x.strip()) for x in line.split(',')]
            if len(vals) >= 9:
                rows.append(vals[:9])
        if len(rows) < 10:
            return None
        arr = np.array(rows, dtype=np.float32)
        return np.concatenate([arr[:, 0:3], arr[:, 6:9]], axis=1)
    except:
        return None

def parse_filename(fpath):
    base = os.path.basename(fpath).replace('.txt', '')
    if 'readme' in base.lower() or 'desktop' in base.lower():
        return None, None
    parts = base.split('_')
    if len(parts) < 2:
        return None, None
    act_str  = parts[0].upper()
    subj_str = parts[1].upper()
    if act_str.startswith('F'):
        label = 1
    elif act_str.startswith('D'):
        label = 0
    else:
        return None, None
    if subj_str.startswith('SA'):
        try: subj_id = int(subj_str[2:])
        except: return None, None
    elif subj_str.startswith('SE'):
        try: subj_id = int(subj_str[2:]) + 23
        except: return None, None
    else:
        return None, None
    return subj_id, label

def build_model(input_shape=(WINDOW_SIZE, N_FEATURES)):
    inputs = Input(shape=input_shape)
    x = Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.4)(x)
    x = Conv1D(64, 5, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.4)(x)
    x = Conv1D(128, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.4)(x)
    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(0.4)(x)
    x = LSTM(32)(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(0.001),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.Precision(name='precision')]
    )
    return model

# ── LOAD DATA ────────────────────────────────────────────────
print(f'\nScanning: {SISFALL_DIR}')
all_files = glob.glob(os.path.join(SISFALL_DIR, '**', '*.txt'), recursive=True)
all_files = [f for f in all_files if 'readme' not in f.lower()
             and 'desktop' not in f.lower()]
print(f'Found {len(all_files)} .txt files')

subject_data = {}
skipped = loaded = 0

for fpath in sorted(all_files):
    subj_id, label = parse_filename(fpath)
    if subj_id is None:
        skipped += 1
        continue
    raw = load_sisfall_file(fpath)
    if raw is None or raw.shape[0] < WINDOW_SIZE:
        skipped += 1
        continue
    resampled = resample_signal(raw, ORIG_HZ, TARGET_HZ).astype(np.float32)
    if resampled.shape[0] < WINDOW_SIZE:
        skipped += 1
        continue
    if subj_id not in subject_data:
        subject_data[subj_id] = {'X': [], 'y': []}
    for w in sliding_window(resampled, WINDOW_SIZE, STEP_SIZE):
        if w.shape == (WINDOW_SIZE, N_FEATURES):
            subject_data[subj_id]['X'].append(w)
            subject_data[subj_id]['y'].append(label)
    loaded += 1

print(f'Loaded: {loaded} | Skipped: {skipped}')

subjects = sorted(subject_data.keys())
for s in subjects:
    subject_data[s]['X'] = np.array(subject_data[s]['X'], dtype=np.float32)
    subject_data[s]['y'] = np.array(subject_data[s]['y'], dtype=np.int32)

total_w = sum(len(v['y']) for v in subject_data.values())
total_f = sum(int(v['y'].sum()) for v in subject_data.values())
total_a = total_w - total_f
print(f'Total windows: {total_w:,} | Falls: {total_f:,} | ADL: {total_a:,}')
print(f'Subjects: {len(subjects)} | Ratio 1:{total_a/total_f:.1f}')

# ── LOSO ─────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  LOSO — SisFall @ {TARGET_HZ}Hz")
print(f"{'='*55}\n")

loso_results    = []
all_y_true      = []
all_y_pred_prob = []
best_auc        = 0.0
best_fold_subj  = None
best_model_path = os.path.join(OUTPUT_DIR, 'sisfall_best_fold.keras')
scalers         = {}

for test_subj in subjects:
    grp = 'Young' if test_subj <= 23 else 'Elderly'
    print(f'\n--- Fold: S{test_subj:02d} ({grp}) ---')

    X_test = subject_data[test_subj]['X']
    y_test = subject_data[test_subj]['y']

    if y_test.sum() == 0:
        print('  Skipped: no falls')
        loso_results.append({'subject': test_subj, 'group': grp,
                             'accuracy': None, 'recall': None,
                             'precision': None, 'f1': None,
                             'auc': None, 'note': 'No falls'})
        continue

    train_subjs = [s for s in subjects if s != test_subj]
    X_all = np.concatenate([subject_data[s]['X'] for s in train_subjs])
    y_all = np.concatenate([subject_data[s]['y'] for s in train_subjs])

   
    X_train, X_val, y_train, y_val = train_test_split(
        X_all, y_all, test_size=VAL_SPLIT,
        random_state=42, stratify=y_all
    )

    scaler  = StandardScaler()
    n_tr, T, F = X_train.shape
    X_tr_sc = scaler.fit_transform(X_train.reshape(-1,F)).reshape(n_tr,T,F)
    X_va_sc = scaler.transform(X_val.reshape(-1,F)).reshape(len(X_val),T,F)
    X_te_sc = scaler.transform(X_test.reshape(-1,F)).reshape(len(X_test),T,F)

    
    scalers[test_subj] = scaler

    n_fall = int(y_train.sum())
    n_adl  = len(y_train) - n_fall
    cw = {0: 1.0, 1: float(n_adl / n_fall)}
    print(f'  Train:{len(X_train):,} Val:{len(X_val):,} '
          f'Test:{len(X_test):,} | CW_fall:{cw[1]:.2f}')

    K.clear_session()
    model = build_model()

    fold_ckpt = os.path.join(OUTPUT_DIR, f'fold_S{test_subj:02d}.keras')
    cbs = [
        EarlyStopping(monitor='val_auc', patience=PATIENCE,
                      restore_best_weights=True, mode='max', verbose=1),
        ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5,
                          mode='max', min_lr=1e-6, verbose=1),
        ModelCheckpoint(fold_ckpt, monitor='val_auc',
                        save_best_only=True, mode='max', verbose=0),
    ]

    model.fit(X_tr_sc, y_train,
              validation_data=(X_va_sc, y_val),
              epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
              class_weight=cw, callbacks=cbs, verbose=1)

    y_prob = model.predict(X_te_sc, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    acc  = float(accuracy_score(y_test, y_pred))
    rec  = float(recall_score(y_test, y_pred, zero_division=0))
    prec = float(precision_score(y_test, y_pred, zero_division=0))
    f1   = float(f1_score(y_test, y_pred, zero_division=0))
    try:    auc = float(roc_auc_score(y_test, y_prob))
    except: auc = None

    auc_str = f'{auc:.4f}' if auc else 'N/A'
    print(f'  Acc={acc:.4f} Rec={rec:.4f} Prec={prec:.4f} F1={f1:.4f} AUC={auc_str}')

    loso_results.append({'subject': test_subj, 'group': grp,
                         'accuracy': acc, 'recall': rec,
                         'precision': prec, 'f1': f1,
                         'auc': auc, 'note': ''})
    all_y_true.extend(y_test.tolist())
    all_y_pred_prob.extend(y_prob.tolist())

   
    if auc and auc > best_auc:
        best_auc       = auc
        best_fold_subj = test_subj
        shutil.copy(fold_ckpt, best_model_path)
        print(f'  ★ Best AUC={best_auc:.4f} (S{test_subj:02d}) saved!')

print(f'\nLOSO done! Best: S{best_fold_subj:02d} AUC={best_auc:.4f}')

# ── AGGREGATE + THRESHOLD ────────────────────────────────────
all_y_true      = np.array(all_y_true)
all_y_pred_prob = np.array(all_y_pred_prob)

thresholds = np.arange(0.25, 0.86, 0.05)
th_results = []
for th in thresholds:
    yp = (all_y_pred_prob >= th).astype(int)
    th_results.append({
        'threshold':    round(float(th), 2),
        'recall':       float(recall_score(all_y_true, yp, zero_division=0)),
        'precision':    float(precision_score(all_y_true, yp, zero_division=0)),
        'f1':           float(f1_score(all_y_true, yp, zero_division=0)),
        'false_alarms': int(((yp==1)&(all_y_true==0)).sum()),
        'missed_falls': int(((yp==0)&(all_y_true==1)).sum()),
    })

eligible = [r for r in th_results if r['recall'] >= 0.90]
best_th  = max(eligible, key=lambda x: x['f1']) if eligible else \
           max(th_results, key=lambda x: x['f1'])

all_y_pred = (all_y_pred_prob >= best_th['threshold']).astype(int)
agg_acc  = float(accuracy_score(all_y_true, all_y_pred))
agg_rec  = float(recall_score(all_y_true, all_y_pred, zero_division=0))
agg_prec = float(precision_score(all_y_true, all_y_pred, zero_division=0))
agg_f1   = float(f1_score(all_y_true, all_y_pred, zero_division=0))
agg_auc  = float(roc_auc_score(all_y_true, all_y_pred_prob))

valid = [r for r in loso_results if r['auc'] is not None]
young = [r for r in valid if r['group'] == 'Young']
elder = [r for r in valid if r['group'] == 'Elderly']

print(f"\n{'='*55}")
print(f"  FINAL RESULTS — SisFall @ {TARGET_HZ}Hz")
print(f"{'='*55}")
print(f"  Accuracy  : {agg_acc*100:.2f}%")
print(f"  Recall    : {agg_rec*100:.2f}%")
print(f"  Precision : {agg_prec*100:.2f}%")
print(f"  F1 Score  : {agg_f1*100:.2f}%")
print(f"  AUC-ROC   : {agg_auc*100:.2f}%")
print(f"  Threshold : {best_th['threshold']}")
if young:
    print(f"\n  Young  : AUC={np.mean([r['auc'] for r in young])*100:.2f}% "
          f"F1={np.mean([r['f1'] for r in young])*100:.2f}%")
if elder:
    print(f"  Elderly: AUC={np.mean([r['auc'] for r in elder])*100:.2f}% "
          f"F1={np.mean([r['f1'] for r in elder])*100:.2f}%")
print(f"\n{classification_report(all_y_true, all_y_pred, target_names=['ADL','Fall'])}")

# ── PLOTS ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SisFall CNN-LSTM LOSO Results', fontweight='bold')
cm_mat = confusion_matrix(all_y_true, all_y_pred)
im = axes[0].imshow(cm_mat, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['ADL','Fall'])
axes[0].set_yticklabels(['ADL','Fall'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix\nAcc={agg_acc*100:.2f}% F1={agg_f1*100:.2f}%')
plt.colorbar(im, ax=axes[0])
for i in range(2):
    for j in range(2):
        pct = cm_mat[i,j]/cm_mat[i].sum()*100
        axes[0].text(j, i, f'{cm_mat[i,j]:,}\n({pct:.1f}%)',
                     ha='center', va='center', fontsize=12,
                     color='white' if cm_mat[i,j]>cm_mat.max()/2 else 'black')
fpr, tpr, _ = roc_curve(all_y_true, all_y_pred_prob)
axes[1].plot(fpr, tpr, 'steelblue', lw=2,
             label=f'CNN-LSTM LOSO (AUC={agg_auc:.3f})')
axes[1].plot([0,1],[0,1],'r--', label='Random')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'sisfall_results.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved: sisfall_results.png')

# ── SAVE ─────────────────────────────────────────────────────
print('\nSaving model...')
K.clear_session()
final_model = tf.keras.models.load_model(best_model_path)
final_model.save(os.path.join(OUTPUT_DIR,'sisfall_6feat_final.keras'))
final_model.save(os.path.join(OUTPUT_DIR,'sisfall_6feat_final.h5'))

# FIX 4: Best fold ka scaler save karo
best_scaler = scalers[best_fold_subj]
with open(os.path.join(OUTPUT_DIR,'sisfall_6feat_scaler.pkl'),'wb') as f:
    pickle.dump(best_scaler, f)
with open(os.path.join(OUTPUT_DIR,'sisfall_6feat_all_scalers.pkl'),'wb') as f:
    pickle.dump(scalers, f)

config = {
    'dataset': 'SisFall',
    'features': 'Acc1_xyz + Gyr_xyz',
    'feature_order': ['Acc1_x','Acc1_y','Acc1_z','Gyr_x','Gyr_y','Gyr_z'],
    'n_features': N_FEATURES,
    'sampling_rate': TARGET_HZ,
    'window_size': WINDOW_SIZE,
    'step_size': STEP_SIZE,
    'threshold': best_th['threshold'],
    'best_fold': best_fold_subj,
    'aggregate': {
        'auc': round(agg_auc,4), 'f1': round(agg_f1,4),
        'recall': round(agg_rec,4), 'precision': round(agg_prec,4),
        'accuracy': round(agg_acc,4)
    },
    'young_mean_auc': round(float(np.mean([r['auc'] for r in young])),4) if young else None,
    'elderly_mean_auc': round(float(np.mean([r['auc'] for r in elder])),4) if elder else None,
}
with open(os.path.join(OUTPUT_DIR,'sisfall_6feat_config.json'),'w') as f:
    json.dump(config, f, indent=2)

pd.DataFrame(loso_results).to_csv(
    os.path.join(OUTPUT_DIR,'loso_results_sisfall.csv'), index=False)

print('  ✓ sisfall_6feat_final.keras')
print('  ✓ sisfall_6feat_final.h5')
print('  ✓ sisfall_6feat_scaler.pkl')
print('  ✓ sisfall_6feat_config.json')
print('  ✓ loso_results_sisfall.csv')
print('\nDone! ✓')

TF version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
  DATASET  : SisFall
  FEATURES : Acc1(xyz) + Gyr(xyz) = 6
  SAMPLING : 200Hz → 50Hz
  WINDOW   : 250 samples (5.0s)
  OVERLAP  : 80%

Scanning: /kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset
Found 4505 .txt files
Loaded: 4505 | Skipped: 0
Total windows: 60,894 | Falls: 19,656 | ADL: 41,238
Subjects: 38 | Ratio 1:2.1

  LOSO — SisFall @ 50Hz


--- Fold: S01 (Young) ---
  Train:53,004 Val:5,890 Test:2,000 | CW_fall:2.13
Epoch 1/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.8646 - auc: 0.9112 - loss: 0.5046 - precision: 0.7734 - recall: 0.8253 - val_accuracy: 0.9289 - val_auc: 0.9742 - val_loss: 0.1846 - val_precision: 0.8885 - val_recall: 0.8890 - learning_rate: 0.0010
Epoch 2/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9264 - auc: 0.9730 - loss: 0.2808 - precision: 0.8734

In [16]:
import numpy as np
import pandas as pd
import os, glob, pickle, json, shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization,
                                     MaxPooling1D, Dropout, LSTM, Dense)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import tensorflow.keras.backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, recall_score,
                             precision_score, accuracy_score, roc_curve)

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

# ── CONFIG ───────────────────────────────────────────────────
SISFALL_DIR = '/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset'
OUTPUT_DIR  = '/kaggle/working/'
ORIG_HZ     = 200
TARGET_HZ   = 50
WINDOW_SIZE = 250
STEP_SIZE   = 50
N_FEATURES  = 6
BATCH_SIZE  = 64
MAX_EPOCHS  = 50
PATIENCE    = 10
VAL_SPLIT   = 0.10

# ── HELPERS ──────────────────────────────────────────────────
def resample_signal(signal, orig_hz, target_hz):
    orig_len   = len(signal)
    target_len = int(orig_len * target_hz / orig_hz)
    if target_len < 2:
        return signal
    orig_t   = np.linspace(0, 1, orig_len)
    target_t = np.linspace(0, 1, target_len)
    if signal.ndim == 1:
        return np.interp(target_t, orig_t, signal)
    return np.column_stack([
        np.interp(target_t, orig_t, signal[:, i])
        for i in range(signal.shape[1])
    ])

def sliding_window(signal, window_size, step_size):
    windows, start = [], 0
    while start + window_size <= signal.shape[0]:
        windows.append(signal[start:start + window_size])
        start += step_size
    return windows

def load_sisfall_file(filepath):
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
        rows = []
        for line in lines:
            line = line.strip().rstrip(';').strip()
            if not line:
                continue
            vals = [float(x.strip()) for x in line.split(',')]
            if len(vals) >= 9:
                rows.append(vals[:9])
        if len(rows) < 10:
            return None
        arr = np.array(rows, dtype=np.float32)
        return np.concatenate([arr[:, 0:3], arr[:, 6:9]], axis=1)
    except:
        return None

def parse_filename(fpath):
    base = os.path.basename(fpath).replace('.txt', '')
    if 'readme' in base.lower() or 'desktop' in base.lower():
        return None, None
    parts = base.split('_')
    if len(parts) < 2:
        return None, None
    act_str  = parts[0].upper()
    subj_str = parts[1].upper()
    if act_str.startswith('F'):
        label = 1
    elif act_str.startswith('D'):
        label = 0
    else:
        return None, None
    if subj_str.startswith('SA'):
        try:    subj_id = int(subj_str[2:])
        except: return None, None
    elif subj_str.startswith('SE'):
        try:    subj_id = int(subj_str[2:]) + 23
        except: return None, None
    else:
        return None, None
    return subj_id, label

def build_model(input_shape=(WINDOW_SIZE, N_FEATURES)):
    inputs = Input(shape=input_shape)
    x = Conv1D(32, 7, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.4)(x)
    x = Conv1D(64, 5, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.4)(x)
    x = Conv1D(128, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)
    x = Dropout(0.4)(x)
    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(0.4)(x)
    x = LSTM(32)(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(0.001),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.Precision(name='precision')]
    )
    return model

# ── LOAD DATA ────────────────────────────────────────────────
print(f'\nScanning: {SISFALL_DIR}')
all_files = glob.glob(os.path.join(SISFALL_DIR, '**', '*.txt'), recursive=True)
all_files = [f for f in all_files if 'readme' not in f.lower()
             and 'desktop' not in f.lower()]
print(f'Found {len(all_files)} files')

subject_data = {}
skipped = loaded = 0

for fpath in sorted(all_files):
    subj_id, label = parse_filename(fpath)
    if subj_id is None:
        skipped += 1
        continue
    raw = load_sisfall_file(fpath)
    if raw is None or raw.shape[0] < WINDOW_SIZE:
        skipped += 1
        continue
    resampled = resample_signal(raw, ORIG_HZ, TARGET_HZ).astype(np.float32)
    if resampled.shape[0] < WINDOW_SIZE:
        skipped += 1
        continue
    if subj_id not in subject_data:
        subject_data[subj_id] = {'X': [], 'y': []}
    for w in sliding_window(resampled, WINDOW_SIZE, STEP_SIZE):
        if w.shape == (WINDOW_SIZE, N_FEATURES):
            subject_data[subj_id]['X'].append(w)
            subject_data[subj_id]['y'].append(label)
    loaded += 1

print(f'Loaded: {loaded} | Skipped: {skipped}')

subjects = sorted(subject_data.keys())
for s in subjects:
    subject_data[s]['X'] = np.array(subject_data[s]['X'], dtype=np.float32)
    subject_data[s]['y'] = np.array(subject_data[s]['y'], dtype=np.int32)

total_w = sum(len(v['y']) for v in subject_data.values())
total_f = sum(int(v['y'].sum()) for v in subject_data.values())
total_a = total_w - total_f
print(f'Total: {total_w:,} | Falls: {total_f:,} | ADL: {total_a:,}')
print(f'Subjects: {len(subjects)}')

# ── S01-S18 PREVIOUS RESULTS (already computed) ──────────────
print('\nLoading S01-S18 previous results...')

loso_results = [
    {'subject':1,  'group':'Young','n_test':2000,'n_falls':666,'accuracy':0.9065,'recall':0.9113,'precision':0.8681,'f1':0.8892,'auc':0.9740,'note':''},
    {'subject':2,  'group':'Young','n_test':1999,'n_falls':666,'accuracy':0.8879,'recall':0.8007,'precision':0.9166,'f1':0.8547,'auc':0.9641,'note':''},
    {'subject':3,  'group':'Young','n_test':1993,'n_falls':666,'accuracy':0.9343,'recall':0.9610,'precision':0.8884,'f1':0.9233,'auc':0.9903,'note':''},
    {'subject':4,  'group':'Young','n_test':1996,'n_falls':666,'accuracy':0.9254,'recall':0.9683,'precision':0.8660,'f1':0.9143,'auc':0.9921,'note':''},
    {'subject':5,  'group':'Young','n_test':1994,'n_falls':666,'accuracy':0.9283,'recall':0.9756,'precision':0.8669,'f1':0.9181,'auc':0.9923,'note':''},
    {'subject':6,  'group':'Young','n_test':1989,'n_falls':663,'accuracy':0.9829,'recall':0.9671,'precision':0.9912,'f1':0.9790,'auc':0.9978,'note':''},
    {'subject':7,  'group':'Young','n_test':1994,'n_falls':663,'accuracy':0.9709,'recall':0.9659,'precision':0.9636,'f1':0.9648,'auc':0.9964,'note':''},
    {'subject':8,  'group':'Young','n_test':1994,'n_falls':663,'accuracy':0.9077,'recall':0.9599,'precision':0.8394,'f1':0.8956,'auc':0.9805,'note':''},
    {'subject':9,  'group':'Young','n_test':1991,'n_falls':663,'accuracy':0.9724,'recall':0.9708,'precision':0.9626,'f1':0.9667,'auc':0.9957,'note':''},
    {'subject':10, 'group':'Young','n_test':1988,'n_falls':663,'accuracy':0.9759,'recall':0.9511,'precision':0.9898,'f1':0.9701,'auc':0.9975,'note':''},
    {'subject':11, 'group':'Young','n_test':1990,'n_falls':663,'accuracy':0.9322,'recall':0.8976,'precision':0.9352,'f1':0.9160,'auc':0.9718,'note':''},
    {'subject':12, 'group':'Young','n_test':1992,'n_falls':664,'accuracy':0.9603,'recall':0.9780,'precision':0.9292,'f1':0.9530,'auc':0.9939,'note':''},
    {'subject':13, 'group':'Young','n_test':1993,'n_falls':664,'accuracy':0.9497,'recall':0.9412,'precision':0.9366,'f1':0.9389,'auc':0.9917,'note':''},
    {'subject':15, 'group':'Young','n_test':1890,'n_falls':630,'accuracy':0.9635,'recall':0.9671,'precision':0.9498,'f1':0.9584,'auc':0.9935,'note':''},
    {'subject':16, 'group':'Young','n_test':1995,'n_falls':665,'accuracy':0.9684,'recall':0.9574,'precision':0.9656,'f1':0.9615,'auc':0.9963,'note':''},
    {'subject':17, 'group':'Young','n_test':1967,'n_falls':655,'accuracy':0.9670,'recall':0.9669,'precision':0.9540,'f1':0.9604,'auc':0.9951,'note':''},
    {'subject':18, 'group':'Young','n_test':1995,'n_falls':665,'accuracy':0.9163,'recall':0.9513,'precision':0.8603,'f1':0.9035,'auc':0.9835,'note':''},
]

all_y_true      = []
all_y_pred_prob = []
best_auc        = 0.9978   
best_fold_subj  = 6
scalers         = {}
best_model_path = os.path.join(OUTPUT_DIR, 'sisfall_best_fold.keras')

print(f'Loaded {len(loso_results)} previous results')
print(f'Best AUC so far: {best_auc} (S{best_fold_subj:02d})')


print(f'\n{"="*55}')
print(f'  LOSO Resume from S19')
print(f'{"="*55}\n')

for test_subj in subjects:

    
    if test_subj < 19:
        continue

    grp = 'Young' if test_subj <= 23 else 'Elderly'
    print(f'\n--- Fold: S{test_subj:02d} ({grp}) ---')

    X_test = subject_data[test_subj]['X']
    y_test = subject_data[test_subj]['y']

    if y_test.sum() == 0:
        print('  Skipped: no falls')
        loso_results.append({
            'subject': test_subj, 'group': grp,
            'n_test': len(y_test), 'n_falls': 0,
            'accuracy': None, 'recall': None,
            'precision': None, 'f1': None,
            'auc': None, 'note': 'No falls'
        })
        continue

    train_subjs = [s for s in subjects if s != test_subj]
    X_all = np.concatenate([subject_data[s]['X'] for s in train_subjs])
    y_all = np.concatenate([subject_data[s]['y'] for s in train_subjs])

    X_train, X_val, y_train, y_val = train_test_split(
        X_all, y_all, test_size=VAL_SPLIT,
        random_state=42, stratify=y_all
    )

    scaler  = StandardScaler()
    n_tr, T, F = X_train.shape
    X_tr_sc = scaler.fit_transform(X_train.reshape(-1,F)).reshape(n_tr,T,F)
    X_va_sc = scaler.transform(X_val.reshape(-1,F)).reshape(len(X_val),T,F)
    X_te_sc = scaler.transform(X_test.reshape(-1,F)).reshape(len(X_test),T,F)
    scalers[test_subj] = scaler

    n_fall = int(y_train.sum())
    n_adl  = len(y_train) - n_fall
    cw = {0: 1.0, 1: float(n_adl / n_fall)}
    print(f'  Train:{len(X_train):,} Val:{len(X_val):,} '
          f'Test:{len(X_test):,} | CW_fall:{cw[1]:.2f}')

    K.clear_session()
    model = build_model()

    fold_ckpt = os.path.join(OUTPUT_DIR, f'fold_S{test_subj:02d}.keras')
    cbs = [
        EarlyStopping(monitor='val_auc', patience=PATIENCE,
                      restore_best_weights=True, mode='max', verbose=1),
        ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5,
                          mode='max', min_lr=1e-6, verbose=1),
        ModelCheckpoint(fold_ckpt, monitor='val_auc',
                        save_best_only=True, mode='max', verbose=0),
    ]

    model.fit(X_tr_sc, y_train,
              validation_data=(X_va_sc, y_val),
              epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
              class_weight=cw, callbacks=cbs, verbose=1)

    y_prob = model.predict(X_te_sc, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    acc  = float(accuracy_score(y_test, y_pred))
    rec  = float(recall_score(y_test, y_pred, zero_division=0))
    prec = float(precision_score(y_test, y_pred, zero_division=0))
    f1   = float(f1_score(y_test, y_pred, zero_division=0))
    try:    auc = float(roc_auc_score(y_test, y_prob))
    except: auc = None

    auc_str = f'{auc:.4f}' if auc else 'N/A'
    print(f'  Acc={acc:.4f} Rec={rec:.4f} '
          f'Prec={prec:.4f} F1={f1:.4f} AUC={auc_str}')

    loso_results.append({
        'subject': test_subj, 'group': grp,
        'n_test': len(y_test), 'n_falls': int(y_test.sum()),
        'accuracy': acc, 'recall': rec,
        'precision': prec, 'f1': f1,
        'auc': auc, 'note': ''
    })

    all_y_true.extend(y_test.tolist())
    all_y_pred_prob.extend(y_prob.tolist())

    if auc and auc > best_auc:
        best_auc       = auc
        best_fold_subj = test_subj
        shutil.copy(fold_ckpt, best_model_path)
        print(f'  ★ Best AUC={best_auc:.4f} (S{test_subj:02d}) saved!')

print(f'\nLOSO S19-S38 done!')

# ── AGGREGATE (S19-S38 only — S01-S18 probs not available) ───
all_y_true      = np.array(all_y_true)
all_y_pred_prob = np.array(all_y_pred_prob)

if len(all_y_true) > 0:
    thresholds = np.arange(0.25, 0.86, 0.05)
    th_results = []
    for th in thresholds:
        yp = (all_y_pred_prob >= th).astype(int)
        th_results.append({
            'threshold':    round(float(th), 2),
            'recall':       float(recall_score(all_y_true, yp, zero_division=0)),
            'precision':    float(precision_score(all_y_true, yp, zero_division=0)),
            'f1':           float(f1_score(all_y_true, yp, zero_division=0)),
            'false_alarms': int(((yp==1)&(all_y_true==0)).sum()),
            'missed_falls': int(((yp==0)&(all_y_true==1)).sum()),
        })
    eligible = [r for r in th_results if r['recall'] >= 0.90]
    best_th  = max(eligible, key=lambda x: x['f1']) if eligible else \
               max(th_results, key=lambda x: x['f1'])
else:
    best_th = {'threshold': 0.35}

# ── FULL SUMMARY (all 38 subjects) ───────────────────────────
valid  = [r for r in loso_results if r['auc'] is not None]
young  = [r for r in valid if r['group'] == 'Young']
elder  = [r for r in valid if r['group'] == 'Elderly']

mean_acc  = np.mean([r['accuracy']  for r in valid])
mean_rec  = np.mean([r['recall']    for r in valid])
mean_prec = np.mean([r['precision'] for r in valid])
mean_f1   = np.mean([r['f1']        for r in valid])
mean_auc  = np.mean([r['auc']       for r in valid])
std_auc   = np.std( [r['auc']       for r in valid])

print(f'\n{"="*55}')
print(f'  FINAL SUMMARY — SisFall @ {TARGET_HZ}Hz')
print(f'{"="*55}')
print(f'  Mean Accuracy  : {mean_acc*100:.2f}%')
print(f'  Mean Recall    : {mean_rec*100:.2f}%')
print(f'  Mean Precision : {mean_prec*100:.2f}%')
print(f'  Mean F1        : {mean_f1*100:.2f}%')
print(f'  Mean AUC       : {mean_auc*100:.2f}% ± {std_auc*100:.2f}%')

if young:
    print(f'\n  Young  ({len(young)} subj): '
          f'AUC={np.mean([r["auc"] for r in young])*100:.2f}% '
          f'F1={np.mean([r["f1"] for r in young])*100:.2f}%')
if elder:
    print(f'  Elderly({len(elder)} subj): '
          f'AUC={np.mean([r["auc"] for r in elder])*100:.2f}% '
          f'F1={np.mean([r["f1"] for r in elder])*100:.2f}%')

print(f'\n  Best Fold      : S{best_fold_subj:02d} (AUC={best_auc:.4f})')
print(f'  Best Threshold : {best_th["threshold"]}')

# Per subject table
print(f'\n{"─"*65}')
print(f'{"S#":>4}  {"Group":7}  {"Acc":>7}  {"Recall":>7}  '
      f'{"Prec":>7}  {"F1":>7}  {"AUC":>7}')
print(f'{"─"*65}')
for r in loso_results:
    if r['auc'] is None:
        print(f'S{r["subject"]:02d}    {r["group"]:7}  -- skipped --')
    else:
        print(f'S{r["subject"]:02d}    {r["group"]:7}  '
              f'{r["accuracy"]*100:6.2f}%  '
              f'{r["recall"]*100:6.2f}%  '
              f'{r["precision"]*100:6.2f}%  '
              f'{r["f1"]*100:6.2f}%  '
              f'{r["auc"]*100:6.2f}%')

# ── SAVE RESULTS CSV ─────────────────────────────────────────
pd.DataFrame(loso_results).to_csv(
    os.path.join(OUTPUT_DIR, 'loso_results_sisfall_complete.csv'), index=False)
print('\n  ✓ loso_results_sisfall_complete.csv saved!')

# ── SAVE FINAL MODEL ─────────────────────────────────────────
if os.path.exists(best_model_path):
    print('\nSaving final model...')
    K.clear_session()
    final_model = tf.keras.models.load_model(best_model_path)
    final_model.save(os.path.join(OUTPUT_DIR,'sisfall_6feat_final.keras'))
    final_model.save(os.path.join(OUTPUT_DIR,'sisfall_6feat_final.h5'))

    if best_fold_subj in scalers:
        best_scaler = scalers[best_fold_subj]
    else:
        # Use last available scaler
        best_scaler = list(scalers.values())[-1]

    with open(os.path.join(OUTPUT_DIR,'sisfall_6feat_scaler.pkl'),'wb') as f:
        pickle.dump(best_scaler, f)
    with open(os.path.join(OUTPUT_DIR,'sisfall_6feat_all_scalers.pkl'),'wb') as f:
        pickle.dump(scalers, f)

    config = {
        'dataset':       'SisFall',
        'features':      'Acc1_xyz + Gyr_xyz',
        'feature_order': ['Acc1_x','Acc1_y','Acc1_z','Gyr_x','Gyr_y','Gyr_z'],
        'n_features':    N_FEATURES,
        'sampling_rate': TARGET_HZ,
        'window_size':   WINDOW_SIZE,
        'step_size':     STEP_SIZE,
        'threshold':     best_th['threshold'],
        'best_fold':     best_fold_subj,
        'mean_auc':      round(float(mean_auc), 4),
        'mean_f1':       round(float(mean_f1),  4),
        'mean_recall':   round(float(mean_rec), 4),
        'young_mean_auc': round(float(np.mean([r['auc'] for r in young])),4) if young else None,
        'elderly_mean_auc': round(float(np.mean([r['auc'] for r in elder])),4) if elder else None,
    }
    with open(os.path.join(OUTPUT_DIR,'sisfall_6feat_config.json'),'w') as f:
        json.dump(config, f, indent=2)

    print('  ✓ sisfall_6feat_final.keras')
    print('  ✓ sisfall_6feat_final.h5')
    print('  ✓ sisfall_6feat_scaler.pkl')
    print('  ✓ sisfall_6feat_config.json')

print(f'\nAll done! Output: {OUTPUT_DIR}')

TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Scanning: /kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset
Found 4505 files
Loaded: 4505 | Skipped: 0
Total: 60,894 | Falls: 19,656 | ADL: 41,238
Subjects: 38

Loading S01-S18 previous results...
Loaded 17 previous results
Best AUC so far: 0.9978 (S06)

  LOSO Resume from S19


--- Fold: S19 (Young) ---
  Train:53,019 Val:5,892 Test:1,983 | CW_fall:2.13
Epoch 1/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - accuracy: 0.8498 - auc: 0.8960 - loss: 0.5428 - precision: 0.7494 - recall: 0.8072 - val_accuracy: 0.9190 - val_auc: 0.9715 - val_loss: 0.2151 - val_precision: 0.8643 - val_recall: 0.8859 - learning_rate: 0.0010
Epoch 2/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9148 - auc: 0.9690 - loss: 0.3050 - precision: 0.8489 - recall: 0.8918 - val_accuracy: 0.9362 - val_auc: 0.9823 - val_loss: 0.176

  ✓ sisfall_6feat_final.keras
  ✓ sisfall_6feat_final.h5
  ✓ sisfall_6feat_scaler.pkl
  ✓ sisfall_6feat_config.json

All done! Output: /kaggle/working/


In [19]:
test_subj = 14
grp = 'Young'
print(f'--- Training S{test_subj:02d} ({grp}) ---')

X_test = subject_data[test_subj]['X']
y_test = subject_data[test_subj]['y']

train_subjs = [s for s in subjects if s != test_subj]
X_all = np.concatenate([subject_data[s]['X'] for s in train_subjs])
y_all = np.concatenate([subject_data[s]['y'] for s in train_subjs])

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=VAL_SPLIT,
    random_state=42, stratify=y_all
)

scaler = StandardScaler()
n_tr, T, F = X_train.shape
X_tr_sc = scaler.fit_transform(X_train.reshape(-1,F)).reshape(n_tr,T,F)
X_va_sc = scaler.transform(X_val.reshape(-1,F)).reshape(len(X_val),T,F)
X_te_sc = scaler.transform(X_test.reshape(-1,F)).reshape(len(X_test),T,F)

n_fall = int(y_train.sum())
n_adl  = len(y_train) - n_fall
cw = {0: 1.0, 1: float(n_adl / n_fall)}
print(f'  Train:{len(X_train):,} Val:{len(X_val):,} Test:{len(X_test):,} | CW_fall:{cw[1]:.2f}')

K.clear_session()
model = build_model()

fold_ckpt = os.path.join(OUTPUT_DIR, f'fold_S{test_subj:02d}.keras')
cbs = [
    EarlyStopping(monitor='val_auc', patience=PATIENCE,
                  restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5,
                      mode='max', min_lr=1e-6, verbose=1),
    ModelCheckpoint(fold_ckpt, monitor='val_auc',
                    save_best_only=True, mode='max', verbose=0),
]

model.fit(X_tr_sc, y_train,
          validation_data=(X_va_sc, y_val),
          epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
          class_weight=cw, callbacks=cbs, verbose=1)

y_prob = model.predict(X_te_sc, verbose=0).flatten()
y_pred = (y_prob >= 0.5).astype(int)

acc  = float(accuracy_score(y_test, y_pred))
rec  = float(recall_score(y_test, y_pred, zero_division=0))
prec = float(precision_score(y_test, y_pred, zero_division=0))
f1   = float(f1_score(y_test, y_pred, zero_division=0))
try:    auc = float(roc_auc_score(y_test, y_prob))
except: auc = None

print(f'\n  S14 Results:')
print(f'  Acc={acc:.4f} Rec={rec:.4f} Prec={prec:.4f} F1={f1:.4f} AUC={auc:.4f}')


s14_result = {
    'subject': 14, 'group': 'Young',
    'n_test': len(y_test), 'n_falls': int(y_test.sum()),
    'accuracy': acc, 'recall': rec,
    'precision': prec, 'f1': f1,
    'auc': auc, 'note': ''
}


insert_idx = next(i for i,r in enumerate(loso_results) if r['subject'] == 15)
loso_results.insert(insert_idx, s14_result)
print(f'\n  S14 added to results at index {insert_idx}')
print(f'  Total results: {len(loso_results)}')

--- Training S14 (Young) ---
  Train:53,015 Val:5,891 Test:1,988 | CW_fall:2.13
Epoch 1/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 16s 15ms/step - accuracy: 0.8568 - auc: 0.9031 - loss: 0.5224 - precision: 0.7512 - recall: 0.8253 - val_accuracy: 0.9127 - val_auc: 0.9703 - val_loss: 0.2296 - val_precision: 0.8332 - val_recall: 0.9092 - learning_rate: 0.0010
Epoch 2/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9049 - auc: 0.9631 - loss: 0.3333 - precision: 0.8198 - recall: 0.8990 - val_accuracy: 0.9372 - val_auc: 0.9826 - val_loss: 0.1769 - val_precision: 0.8886 - val_recall: 0.9188 - learning_rate: 0.0010
Epoch 3/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.9336 - auc: 0.9772 - loss: 0.2565 - precision: 0.8824 - recall: 0.9120 - val_accuracy: 0.9542 - val_auc: 0.9885 - val_loss: 0.1441 - val_precision: 0.9134 - val_recall: 0.9464 - learning_rate: 0.0010
Epoch 4/50
829/829 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.9443 - auc: 0.9828 - loss: 0.2224 - precision: 0

In [20]:
# Updated summary
valid  = [r for r in loso_results if r['auc'] is not None]
young  = [r for r in valid if r['group'] == 'Young']
elder  = [r for r in valid if r['group'] == 'Elderly']

mean_auc  = np.mean([r['auc']       for r in valid])
mean_f1   = np.mean([r['f1']        for r in valid])
mean_rec  = np.mean([r['recall']    for r in valid])
mean_prec = np.mean([r['precision'] for r in valid])
mean_acc  = np.mean([r['accuracy']  for r in valid])
std_auc   = np.std( [r['auc']       for r in valid])

print(f'\n{"="*50}')
print(f'  UPDATED FINAL SUMMARY (S14 included)')
print(f'{"="*50}')
print(f'  Valid subjects : {len(valid)} (was {len(valid)-1})')
print(f'  Mean Accuracy  : {mean_acc*100:.2f}%')
print(f'  Mean Recall    : {mean_rec*100:.2f}%')
print(f'  Mean Precision : {mean_prec*100:.2f}%')
print(f'  Mean F1        : {mean_f1*100:.2f}%')
print(f'  Mean AUC       : {mean_auc*100:.2f}% ± {std_auc*100:.2f}%')

if young:
    print(f'\n  Young  ({len(young)} subj): '
          f'AUC={np.mean([r["auc"] for r in young])*100:.2f}%  '
          f'F1={np.mean([r["f1"] for r in young])*100:.2f}%')
if elder:
    print(f'  Elderly({len(elder)} subj): '
          f'AUC={np.mean([r["auc"] for r in elder])*100:.2f}%  '
          f'F1={np.mean([r["f1"] for r in elder])*100:.2f}%')


pd.DataFrame(loso_results).to_csv(
    os.path.join(OUTPUT_DIR, 'loso_results_sisfall_complete.csv'), index=False)
print(f'\n  ✓ CSV updated with S14!')


  UPDATED FINAL SUMMARY (S14 included)
  Valid subjects : 24 (was 23)
  Mean Accuracy  : 94.35%
  Mean Recall    : 94.46%
  Mean Precision : 92.24%
  Mean F1        : 93.25%
  Mean AUC       : 98.81% ± 1.20%

  Young  (23 subj): AUC=98.99%  F1=93.59%
  Elderly(1 subj): AUC=94.85%  F1=85.43%

  ✓ CSV updated with S14!


In [22]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import os

OUTPUT_DIR = '/kaggle/working/'

# ── 1. CONFUSION MATRIX (Aggregate) ──────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('SisFall CNN-LSTM — Aggregate Results @ 50Hz', 
             fontsize=14, fontweight='bold')

cm = confusion_matrix(all_y_true, (all_y_pred_prob >= best_th['threshold']).astype(int))
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['ADL','Fall'], fontsize=12)
axes[0].set_yticklabels(['ADL','Fall'], fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title(f'Confusion Matrix (S19-S38)\nThreshold={best_th["threshold"]}')
plt.colorbar(im, ax=axes[0])
for i in range(2):
    for j in range(2):
        pct = cm[i,j] / cm[i].sum() * 100
        axes[0].text(j, i, f'{cm[i,j]:,}\n({pct:.1f}%)',
                     ha='center', va='center', fontsize=13, fontweight='bold',
                     color='white' if cm[i,j] > cm.max()/2 else 'black')

# ROC Curve
fpr, tpr, _ = roc_curve(all_y_true, all_y_pred_prob)
auc_val = roc_auc_score(all_y_true, all_y_pred_prob)
axes[1].plot(fpr, tpr, color='steelblue', lw=2.5,
             label=f'CNN-LSTM (AUC={auc_val:.3f})')
axes[1].plot([0,1],[0,1], 'r--', lw=1.5, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve (S19-S38)')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sisfall_confusion_roc.png'), 
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved: sisfall_confusion_roc.png')

# ── 2. PER-SUBJECT AUC BAR CHART ─────────────────────────────
valid_r = [r for r in loso_results if r['auc'] is not None]
subj_labels = [f"S{r['subject']:02d}" for r in valid_r]
aucs   = [r['auc']       for r in valid_r]
f1s    = [r['f1']        for r in valid_r]
recs   = [r['recall']    for r in valid_r]
precs  = [r['precision'] for r in valid_r]
accs   = [r['accuracy']  for r in valid_r]
grps   = [r['group']     for r in valid_r]

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
fig.suptitle('SisFall LOSO Per-Subject Results @ 50Hz', 
             fontsize=14, fontweight='bold')

# All metrics
x = np.arange(len(valid_r))
w = 0.18
axes[0].bar(x - 2*w, accs,  w, label='Accuracy',  color='#4472C4', edgecolor='white')
axes[0].bar(x - 1*w, recs,  w, label='Recall',    color='#FF0000', edgecolor='white')
axes[0].bar(x,       precs, w, label='Precision', color='#FFC000', edgecolor='white')
axes[0].bar(x + 1*w, f1s,   w, label='F1',        color='#70AD47', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(subj_labels, fontsize=8, rotation=45)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_ylim(0.7, 1.05)
axes[0].set_title('All Metrics per Subject')
axes[0].legend(fontsize=10)
axes[0].grid(axis='y', alpha=0.3)
# Elderly boundary
for i, g in enumerate(grps):
    if g == 'Elderly':
        axes[0].axvline(i - 0.5, color='gray', linestyle='--', alpha=0.6, lw=1.5)
        axes[0].text(i, 0.72, 'Elderly→', fontsize=8, color='gray')
        break

# AUC bar chart with colors
bar_colors = ['#70AD47' if a >= 0.97 else ('#FFC000' if a >= 0.95 else '#FF0000')
              for a in aucs]
bars = axes[1].bar(x, aucs, color=bar_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val*100:.1f}', ha='center', va='bottom', 
                 fontsize=7, fontweight='bold')
axes[1].axhline(0.97, color='black', linestyle='--', alpha=0.5, lw=1.5, label='AUC=97%')
axes[1].axhline(np.mean(aucs), color='blue', linestyle='-.', alpha=0.7, lw=2,
                label=f'Mean={np.mean(aucs)*100:.2f}%')
axes[1].set_xticks(x)
axes[1].set_xticklabels(subj_labels, fontsize=8, rotation=45)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_ylim(0.85, 1.02)
axes[1].set_title('AUC-ROC per Subject')
axes[1].legend(handles=[
    plt.Rectangle((0,0),1,1, color='#70AD47', label='AUC ≥ 97%'),
    plt.Rectangle((0,0),1,1, color='#FFC000', label='AUC ≥ 95%'),
    plt.Rectangle((0,0),1,1, color='#FF0000', label='AUC < 95%'),
    plt.Line2D([0],[0], color='blue', linestyle='-.', label=f'Mean={np.mean(aucs)*100:.2f}%'),
], fontsize=9, loc='lower right')
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sisfall_per_subject.png'), 
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved: sisfall_per_subject.png')

# ── 3. YOUNG vs ELDERLY ───────────────────────────────────────
young_r = [r for r in valid_r if r['group'] == 'Young']
elder_r = [r for r in valid_r if r['group'] == 'Elderly']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Young vs Elderly — Performance Comparison', 
             fontsize=14, fontweight='bold')

# Group comparison bar
metrics = ['AUC', 'F1', 'Recall', 'Precision', 'Accuracy']
young_vals = [
    np.mean([r['auc'] for r in young_r]),
    np.mean([r['f1'] for r in young_r]),
    np.mean([r['recall'] for r in young_r]),
    np.mean([r['precision'] for r in young_r]),
    np.mean([r['accuracy'] for r in young_r]),
]
elder_vals = [
    np.mean([r['auc'] for r in elder_r]),
    np.mean([r['f1'] for r in elder_r]),
    np.mean([r['recall'] for r in elder_r]),
    np.mean([r['precision'] for r in elder_r]),
    np.mean([r['accuracy'] for r in elder_r]),
] if elder_r else [0]*5

x2 = np.arange(len(metrics))
axes[0].bar(x2 - 0.2, young_vals, 0.4, label=f'Young (n={len(young_r)})', 
            color='#4472C4', edgecolor='white')
axes[0].bar(x2 + 0.2, elder_vals, 0.4, label=f'Elderly (n={len(elder_r)})', 
            color='#FFC000', edgecolor='white')
for i, (yv, ev) in enumerate(zip(young_vals, elder_vals)):
    axes[0].text(i - 0.2, yv + 0.003, f'{yv*100:.1f}%', 
                 ha='center', fontsize=9, fontweight='bold', color='#4472C4')
    if ev > 0:
        axes[0].text(i + 0.2, ev + 0.003, f'{ev*100:.1f}%', 
                     ha='center', fontsize=9, fontweight='bold', color='#FFC000')
axes[0].set_xticks(x2)
axes[0].set_xticklabels(metrics, fontsize=11)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_ylim(0.8, 1.05)
axes[0].set_title('Young vs Elderly — All Metrics')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# AUC distribution
young_aucs = [r['auc'] for r in young_r]
axes[1].hist(young_aucs, bins=8, color='#4472C4', alpha=0.7, 
             edgecolor='white', label=f'Young (n={len(young_r)})')
axes[1].axvline(np.mean(young_aucs), color='#4472C4', linestyle='--', lw=2,
                label=f'Young Mean: {np.mean(young_aucs)*100:.2f}%')
if elder_r:
    axes[1].axvline(elder_r[0]['auc'], color='#FFC000', linestyle='-', lw=3,
                    label=f'Elderly S29: {elder_r[0]["auc"]*100:.2f}%')
axes[1].set_xlabel('AUC-ROC', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('AUC Distribution')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sisfall_young_vs_elderly.png'), 
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved: sisfall_young_vs_elderly.png')

# ── 4. THRESHOLD TUNING PLOT ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold Tuning — SisFall @ 50Hz', fontweight='bold')

ths  = [r['threshold']    for r in th_results]
rcs  = [r['recall']       for r in th_results]
prs  = [r['precision']    for r in th_results]
f1v  = [r['f1']           for r in th_results]
fas  = [r['false_alarms'] for r in th_results]
mfs  = [r['missed_falls'] for r in th_results]

axes[0].plot(ths, rcs, 'r-o', lw=2, label='Recall',    markersize=6)
axes[0].plot(ths, prs, 'b-o', lw=2, label='Precision', markersize=6)
axes[0].plot(ths, f1v, 'g-o', lw=2, label='F1',        markersize=6)
axes[0].axvline(best_th['threshold'], color='green', lw=2.5,
                label=f"Best ({best_th['threshold']})")
axes[0].axhline(0.90, color='red', linestyle=':', alpha=0.7, label='Recall=90%')
axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Metrics vs Threshold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

ax2 = axes[1].twinx()
axes[1].bar(ths, fas, width=0.04, alpha=0.6, color='salmon', label='False Alarms')
ax2.plot(ths, mfs, 'k-o', lw=2, label='Missed Falls', markersize=6)
axes[1].axvline(best_th['threshold'], color='green', lw=2.5)
axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('False Alarms', color='salmon', fontsize=12)
ax2.set_ylabel('Missed Falls', fontsize=12)
axes[1].set_title('False Alarms vs Missed Falls')
lines1, labs1 = axes[1].get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labs1 + labs2, fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sisfall_threshold.png'), 
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved: sisfall_threshold.png')

# ── 5. SUMMARY STATS TABLE PLOT ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

valid_sorted = sorted(valid_r, key=lambda x: x['subject'])
table_data = []
for r in valid_sorted:
    table_data.append([
        f"S{r['subject']:02d}",
        r['group'],
        f"{r['accuracy']*100:.2f}%",
        f"{r['recall']*100:.2f}%",
        f"{r['precision']*100:.2f}%",
        f"{r['f1']*100:.2f}%",
        f"{r['auc']*100:.2f}%",
    ])

# Add mean row
table_data.append([
    'MEAN', '—',
    f"{np.mean(accs)*100:.2f}%",
    f"{np.mean(recs)*100:.2f}%",
    f"{np.mean(precs)*100:.2f}%",
    f"{np.mean(f1s)*100:.2f}%",
    f"{np.mean(aucs)*100:.2f}%",
])

col_labels = ['Subject', 'Group', 'Accuracy', 'Recall', 'Precision', 'F1', 'AUC']
table = ax.table(cellText=table_data, colLabels=col_labels,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.4)

# Header styling
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#2E75B6')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Row colors
for i in range(1, len(table_data)+1):
    row_data = table_data[i-1]
    is_mean = row_data[0] == 'MEAN'
    is_elderly = row_data[1] == 'Elderly'
    for j in range(len(col_labels)):
        if is_mean:
            table[i, j].set_facecolor('#1F3864')
            table[i, j].set_text_props(color='white', fontweight='bold')
        elif is_elderly:
            table[i, j].set_facecolor('#FFF2CC')
        elif i % 2 == 0:
            table[i, j].set_facecolor('#F2F2F2')

ax.set_title('SisFall CNN-LSTM — Complete LOSO Results @ 50Hz',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sisfall_results_table.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved: sisfall_results_table.png')

print('\n✅ All plots saved!')
print('Files:')
for f in ['sisfall_confusion_roc.png', 'sisfall_per_subject.png',
          'sisfall_young_vs_elderly.png', 'sisfall_threshold.png',
          'sisfall_results_table.png']:
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(path):
        kb = os.path.getsize(path)/1024
        print(f'  ✓ {f:45s} {kb:.1f} KB')


Saved: sisfall_confusion_roc.png
Saved: sisfall_per_subject.png
Saved: sisfall_young_vs_elderly.png
Saved: sisfall_threshold.png
Saved: sisfall_results_table.png

✅ All plots saved!
Files:
  ✓ sisfall_confusion_roc.png                     124.9 KB
  ✓ sisfall_per_subject.png                       131.0 KB
  ✓ sisfall_young_vs_elderly.png                  97.6 KB
  ✓ sisfall_threshold.png                         138.7 KB
  ✓ sisfall_results_table.png                     225.9 KB


In [4]:
import numpy as np
import pandas as pd
import pickle, os, glob
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                             precision_score, accuracy_score,
                             classification_report)

SISFALL_DIR    = '/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset'
FALLALLD_MODEL = '/kaggle/input/models/mandavisingh/fallaiid/keras/default/1/fallalld_correct_final.keras'

TARGET_HZ   = 50
WINDOW_SIZE = 250
STEP_SIZE   = 50
N_FEATURES  = 6

def resample_signal(signal, orig_hz, target_hz):
    orig_len   = len(signal)
    target_len = int(orig_len * target_hz / orig_hz)
    if target_len < 2: return signal
    orig_t   = np.linspace(0, 1, orig_len)
    target_t = np.linspace(0, 1, target_len)
    if signal.ndim == 1:
        return np.interp(target_t, orig_t, signal)
    return np.column_stack([
        np.interp(target_t, orig_t, signal[:, i])
        for i in range(signal.shape[1])
    ])

def sliding_window(signal, window_size, step_size):
    windows, T, start = [], signal.shape[0], 0
    while start + window_size <= T:
        windows.append(signal[start:start + window_size])
        start += step_size
    return windows

def load_sisfall_file(fpath):
    """Load SisFall file — handle semicolon at end of each row."""
    try:
        # Replace semicolons then read
        with open(fpath, 'r') as f:
            content = f.read().replace(';', '')
        from io import StringIO
        data = pd.read_csv(StringIO(content), header=None, sep=',')
        arr  = data.values.astype(np.float32)
        acc1 = arr[:, 0:3]   # Acc1
        gyr  = arr[:, 6:9]   # Gyr
        return np.concatenate([acc1, gyr], axis=1)  # (N, 6)
    except Exception as e:
        return None

# Load SisFall
print("Loading SisFall (Acc1+Gyr)...")
all_files = glob.glob(os.path.join(SISFALL_DIR, '*', '*.txt'))
all_files = [f for f in all_files
             if not os.path.basename(f).lower().startswith('readme')]
print(f"Found {len(all_files)} files")

X_all, y_all = [], []
skipped = 0

for fpath in sorted(all_files):
    base  = os.path.basename(fpath).replace('.txt','')
    parts = base.split('_')
    if len(parts) < 2: skipped += 1; continue
    act_str = parts[0]
    if act_str.startswith('F'):   label = 1
    elif act_str.startswith('D'): label = 0
    else: skipped += 1; continue

    combined = load_sisfall_file(fpath)
    if combined is None or len(combined) < WINDOW_SIZE:
        skipped += 1; continue

    resampled = resample_signal(combined, 200, TARGET_HZ).astype(np.float32)
    if resampled.shape[0] < WINDOW_SIZE: skipped += 1; continue

    for w in sliding_window(resampled, WINDOW_SIZE, STEP_SIZE):
        if w.shape == (WINDOW_SIZE, N_FEATURES):
            X_all.append(w)
            y_all.append(label)

X = np.array(X_all, dtype=np.float32)
y = np.array(y_all, dtype=np.int32)
print(f"Skipped: {skipped}")
print(f"Total: {len(y)} | Falls={y.sum()} | ADL={len(y)-y.sum()}")

# SisFall data ka apna scaler
scaler = StandardScaler()
n, T, F = X.shape
X_sc = scaler.fit_transform(X.reshape(-1, F)).reshape(n, T, F)

# Load FallAllD model
print("\nLoading FallAllD model...")
model = tf.keras.models.load_model(FALLALLD_MODEL, compile=False)

# Predict
print("Predicting...")
y_prob = model.predict(X_sc, batch_size=256, verbose=1).flatten()

# Threshold tuning
thresholds = np.arange(0.30, 0.86, 0.05)
best_th, best_f1 = 0.5, 0
for th in thresholds:
    yp = (y_prob >= th).astype(int)
    r  = recall_score(y, yp, zero_division=0)
    f  = f1_score(y, yp, zero_division=0)
    if r >= 0.90 and f > best_f1:
        best_f1, best_th = f, th

y_pred = (y_prob >= best_th).astype(int)

print(f"\n{'='*50}")
print(f"  TEST 1: FallAllD Model → SisFall Data")
print(f"{'='*50}")
print(f"  Threshold : {best_th}")
print(f"  AUC-ROC   : {roc_auc_score(y, y_prob)*100:.2f}%")
print(f"  Recall    : {recall_score(y, y_pred, zero_division=0)*100:.2f}%")
print(f"  Precision : {precision_score(y, y_pred, zero_division=0)*100:.2f}%")
print(f"  F1        : {f1_score(y, y_pred, zero_division=0)*100:.2f}%")
print(f"  Accuracy  : {accuracy_score(y, y_pred)*100:.2f}%")
print(f"\n{classification_report(y, y_pred, target_names=['ADL','Fall'])}")

Loading SisFall (Acc1+Gyr)...
Found 4505 files
Skipped: 0
Total: 60894 | Falls=19656 | ADL=41238

Loading FallAllD model...


2026-03-20 07:46:34.829554: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Predicting...
238/238 ━━━━━━━━━━━━━━━━━━━━ 16s 67ms/step

  TEST 1: FallAllD Model → SisFall Data
  Threshold : 0.5
  AUC-ROC   : 41.96%
  Recall    : 59.86%
  Precision : 28.85%
  F1        : 38.94%
  Accuracy  : 39.40%

              precision    recall  f1-score   support

         ADL       0.61      0.30      0.40     41238
        Fall       0.29      0.60      0.39     19656

    accuracy                           0.39     60894
   macro avg       0.45      0.45      0.39     60894
weighted avg       0.50      0.39      0.40     60894



In [5]:
import numpy as np
import pandas as pd
import pickle, os, glob
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow.keras.backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                             precision_score, accuracy_score,
                             classification_report)

SISFALL_DIR    = '/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset'
FALLALLD_MODEL = '/kaggle/input/models/mandavisingh/fallaiid/keras/default/1/fallalld_correct_final.keras'
OUTPUT_DIR     = '/kaggle/working/'

TARGET_HZ      = 50
WINDOW_SIZE    = 250
STEP_SIZE      = 50
N_FEATURES     = 6
FINETUNE_RATIO = 0.20   # 20% fine-tune, 80% test
BATCH_SIZE     = 32
MAX_EPOCHS     = 20
PATIENCE       = 5
FINETUNE_LR    = 1e-4

def resample_signal(signal, orig_hz, target_hz):
    orig_len   = len(signal)
    target_len = int(orig_len * target_hz / orig_hz)
    if target_len < 2: return signal
    orig_t   = np.linspace(0, 1, orig_len)
    target_t = np.linspace(0, 1, target_len)
    if signal.ndim == 1:
        return np.interp(target_t, orig_t, signal)
    return np.column_stack([
        np.interp(target_t, orig_t, signal[:, i])
        for i in range(signal.shape[1])
    ])

def sliding_window(signal, window_size, step_size):
    windows, T, start = [], signal.shape[0], 0
    while start + window_size <= T:
        windows.append(signal[start:start + window_size])
        start += step_size
    return windows

# Load SisFall
print("Loading SisFall...")
all_files = glob.glob(os.path.join(SISFALL_DIR, '*', '*.txt'))
all_files = [f for f in all_files
             if not os.path.basename(f).lower().startswith('readme')]

X_all, y_all, skipped = [], [], 0
for fpath in sorted(all_files):
    base  = os.path.basename(fpath).replace('.txt','')
    parts = base.split('_')
    if len(parts) < 2: skipped += 1; continue
    act_str = parts[0]
    if act_str.startswith('F'):   label = 1
    elif act_str.startswith('D'): label = 0
    else: skipped += 1; continue
    try:
        with open(fpath, 'r') as f:
            content = f.read().replace(';', '')
        from io import StringIO
        data = pd.read_csv(StringIO(content), header=None, sep=',')
        arr  = data.values.astype(np.float32)
        combined = np.concatenate([arr[:,0:3], arr[:,6:9]], axis=1)
    except: skipped += 1; continue
    resampled = resample_signal(combined, 200, TARGET_HZ).astype(np.float32)
    if resampled.shape[0] < WINDOW_SIZE: skipped += 1; continue
    for w in sliding_window(resampled, WINDOW_SIZE, STEP_SIZE):
        if w.shape == (WINDOW_SIZE, N_FEATURES):
            X_all.append(w); y_all.append(label)

X = np.array(X_all, dtype=np.float32)
y = np.array(y_all, dtype=np.int32)
print(f"Total: {len(y)} | Falls={y.sum()} | ADL={len(y)-y.sum()}")

# Split: 20% finetune, 80% test
X_ft, X_test, y_ft, y_test = train_test_split(
    X, y, test_size=0.80, random_state=42, stratify=y)
print(f"Fine-tune: {len(y_ft)} | Test: {len(y_test)}")

# Scale
scaler    = StandardScaler()
n_ft, T, F = X_ft.shape
X_ft_sc   = scaler.fit_transform(X_ft.reshape(-1,F)).reshape(n_ft,T,F)
n_te      = X_test.shape[0]
X_test_sc = scaler.transform(X_test.reshape(-1,F)).reshape(n_te,T,F)

# ── BEFORE fine-tuning ────────────────────────────────────────
print("\n[BEFORE Fine-tuning]")
K.clear_session()
model = tf.keras.models.load_model(FALLALLD_MODEL, compile=False)
y_prob_b = model.predict(X_test_sc, batch_size=256, verbose=0).flatten()
auc_b = roc_auc_score(y_test, y_prob_b)
y_pred_b = (y_prob_b >= 0.5).astype(int)
print(f"AUC={auc_b*100:.2f}% | F1={f1_score(y_test,y_pred_b,zero_division=0)*100:.2f}%")

# ── FREEZE CNN, FINE-TUNE LSTM+Dense ─────────────────────────
print("\n[Fine-tuning — CNN frozen, LSTM+Dense trainable]")
K.clear_session()
model = tf.keras.models.load_model(FALLALLD_MODEL, compile=False)

frozen, trainable = 0, 0
for layer in model.layers:
    if any(x in layer.name for x in ['conv1d','batch_norm','max_pool']):
        layer.trainable = False; frozen += 1
    else:
        layer.trainable = True; trainable += 1
print(f"Frozen: {frozen} | Trainable: {trainable}")

model.compile(
    optimizer=Adam(FINETUNE_LR),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Recall(name='recall')]
)

n_fall = int(y_ft.sum())
n_adl  = len(y_ft) - n_fall
cw = {0: 1.0, 1: float(n_adl / n_fall)}
print(f"Class weight fall: {cw[1]:.2f}")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_ft_sc, y_ft, test_size=0.15, random_state=42, stratify=y_ft)

cbs = [
    EarlyStopping(monitor='val_auc', patience=PATIENCE,
                  restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=3,
                      mode='max', min_lr=1e-7, verbose=1),
]

model.fit(X_tr, y_tr,
          validation_data=(X_val, y_val),
          epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
          class_weight=cw, callbacks=cbs, verbose=1)

# ── AFTER fine-tuning ─────────────────────────────────────────
print("\n[AFTER Fine-tuning]")
y_prob_a = model.predict(X_test_sc, batch_size=256, verbose=0).flatten()

thresholds = np.arange(0.30, 0.86, 0.05)
best_th, best_f1 = 0.5, 0
for th in thresholds:
    yp = (y_prob_a >= th).astype(int)
    r  = recall_score(y_test, yp, zero_division=0)
    f  = f1_score(y_test, yp, zero_division=0)
    if r >= 0.90 and f > best_f1:
        best_f1, best_th = f, th

y_pred_a = (y_prob_a >= best_th).astype(int)
auc_a    = roc_auc_score(y_test, y_prob_a)

print(f"\n{'='*50}")
print(f"  FallAllD Model → Fine-tune SisFall → Test SisFall")
print(f"{'='*50}")
print(f"  {'Metric':<12} {'Before':>10} {'After':>10} {'Δ':>10}")
print(f"  {'AUC':<12} {auc_b*100:>9.2f}%  "
      f"{auc_a*100:>9.2f}%  "
      f"{(auc_a-auc_b)*100:>+9.2f}%")
print(f"  {'F1':<12} "
      f"{f1_score(y_test,y_pred_b,zero_division=0)*100:>9.2f}%  "
      f"{f1_score(y_test,y_pred_a,zero_division=0)*100:>9.2f}%  "
      f"{(f1_score(y_test,y_pred_a,zero_division=0)-f1_score(y_test,y_pred_b,zero_division=0))*100:>+9.2f}%")
print(f"  {'Recall':<12} "
      f"{recall_score(y_test,y_pred_b,zero_division=0)*100:>9.2f}%  "
      f"{recall_score(y_test,y_pred_a,zero_division=0)*100:>9.2f}%  "
      f"{(recall_score(y_test,y_pred_a,zero_division=0)-recall_score(y_test,y_pred_b,zero_division=0))*100:>+9.2f}%")
print(f"\nThreshold: {best_th}")
print(f"\n{classification_report(y_test, y_pred_a, target_names=['ADL','Fall'])}")

model.save(os.path.join(OUTPUT_DIR, 'finetuned_fallalld_on_sisfall.keras'))
print("Saved: finetuned_fallalld_on_sisfall.keras")

Loading SisFall...
Total: 60894 | Falls=19656 | ADL=41238
Fine-tune: 12178 | Test: 48716

[BEFORE Fine-tuning]
AUC=41.52% | F1=38.76%

[Fine-tuning — CNN frozen, LSTM+Dense trainable]
Frozen: 9 | Trainable: 11
Class weight fall: 2.10
Epoch 1/20
324/324 ━━━━━━━━━━━━━━━━━━━━ 18s 41ms/step - accuracy: 0.6025 - auc: 0.6414 - loss: 1.7077 - recall: 0.6663 - val_accuracy: 0.8030 - val_auc: 0.8502 - val_loss: 0.4512 - val_recall: 0.7102 - learning_rate: 1.0000e-04
Epoch 2/20
324/324 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step - accuracy: 0.8213 - auc: 0.8757 - loss: 0.6031 - recall: 0.7709 - val_accuracy: 0.8495 - val_auc: 0.8925 - val_loss: 0.3917 - val_recall: 0.7542 - learning_rate: 1.0000e-04
Epoch 3/20
324/324 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step - accuracy: 0.8536 - auc: 0.9023 - loss: 0.5269 - recall: 0.7892 - val_accuracy: 0.8610 - val_auc: 0.9122 - val_loss: 0.3656 - val_recall: 0.7678 - learning_rate: 1.0000e-04
Epoch 4/20
324/324 ━━━━━━━━━━━━━━━━━━━━ 13s 39ms/step - accuracy: 0.8660 - auc: 0.

In [2]:
import numpy as np, pandas as pd, glob
from io import StringIO
import glob, os
files = glob.glob('/kaggle/input/datasets/nvnikhil0001/sis-fall-original-dataset/SisFall_dataset/*/*.txt')
with open(files[0], 'r') as f:
    content = f.read().replace(';','')
arr = pd.read_csv(StringIO(content), header=None, sep=',').values.astype(float)
print(f"SisFall Acc:  min={arr[:,0:3].min():.0f}, max={arr[:,0:3].max():.0f}")

SisFall Acc:  min=-537, max=128
